## 🎯 Learning Objectives
* Demonstrate a comprehensive understanding of the end-to-end machine learning workflow.
* Apply data loading, exploration, and preprocessing techniques to real-world datasets.
* Select, train, and evaluate appropriate supervised machine learning models.
* Implement hyperparameter tuning strategies to optimize model performance.
* Utilize scikit-learn Pipelines for efficient and robust model development.
* Interpret model results and discuss potential limitations or next steps.


# ML02-FA: Final Assessment

Welcome to the final assessment for ML-02: Building Your First ML Model! This assessment is designed to evaluate your comprehensive understanding of the supervised machine learning workflow, from data ingestion to model deployment considerations.

Throughout this course, you've learned to:
*   Load and explore datasets using `pandas`.
*   Perform essential data preprocessing steps, including handling missing values, encoding categorical features, and scaling numerical data.
*   Understand and apply various supervised learning algorithms for both regression and classification tasks.
*   Evaluate model performance using appropriate metrics.
*   Optimize models through hyperparameter tuning.
*   Streamline your workflow using `scikit-learn` Pipelines.

This assessment consists of two main parts:
1.  **Review Questions:** A set of conceptual questions to test your theoretical understanding.
2.  **Capstone Project:** A practical, hands-on coding challenge where you will apply your skills to build a machine learning model for a real-world problem.

**Instructions:**
*   Read each question and problem description carefully.
*   For coding tasks, ensure your code is well-commented, readable, and runnable.
*   Demonstrate your understanding of best practices in machine learning development.

Good luck!


## Part 1: Review Questions

Answer the following questions concisely, demonstrating your understanding of the core concepts covered in this course.

1.  **Supervised vs. Unsupervised Learning:** Explain the fundamental difference between supervised and unsupervised learning paradigms. Provide one real-world example for each.

2.  **Bias-Variance Tradeoff:** Describe the bias-variance tradeoff in the context of machine learning models. How does it relate to the concepts of underfitting and overfitting?

3.  **Cross-Validation:** What is k-fold cross-validation, and why is it a crucial technique for robust model evaluation? Illustrate its process with a simple diagram or explanation.

4.  **Feature Scaling:** When and why is feature scaling (e.g., using `StandardScaler` or `MinMaxScaler`) necessary in machine learning? Provide an example of a model that is particularly sensitive to feature scales.

5.  **Hyperparameter Tuning:** Compare and contrast `GridSearchCV` and `RandomizedSearchCV` for hyperparameter tuning. In what scenarios would you prefer one over the other?

6.  **Evaluation Metrics:** You are building a binary classification model to detect a rare disease. Would you primarily rely on accuracy, precision, recall, or F1-score? Justify your choice.

7.  **Scikit-learn Pipelines:** Explain the primary benefits of using `scikit-learn` Pipelines in your machine learning workflow. How do they contribute to cleaner, more robust, and reproducible code?


## Part 2: Capstone Project - Customer Lifetime Value Prediction

**Scenario:**
You are a Data Scientist at a rapidly growing e-commerce company. The marketing team wants to identify high-value customers to tailor personalized campaigns and improve customer retention. Your task is to build a machine learning model that predicts a customer's **Customer Lifetime Value (CLV)** based on their historical purchasing behavior and demographic information.

**Dataset Description (Simulated):**
We will simulate a dataset representing customer profiles. The features will include:
*   `age`: Customer's age (numerical)
*   `total_purchases`: Total number of purchases made by the customer (numerical)
*   `avg_transaction_value`: Average value of each transaction (numerical)
*   `membership_tier`: Customer's membership level (e.g., 'Bronze', 'Silver', 'Gold', 'Platinum') (categorical)
*   `last_purchase_days_ago`: Number of days since the last purchase (numerical, may have missing values for new customers)
*   `region`: Geographic region of the customer (e.g., 'North', 'South', 'East', 'West') (categorical)
*   `clv`: Customer Lifetime Value (target variable, numerical)

**Your Task:**
Develop a machine learning pipeline using `scikit-learn` to predict `clv`. Your solution should include the following steps:

1.  **Data Generation:** Create a synthetic dataset that mimics the description above. Ensure it includes a mix of numerical and categorical features, and introduce some missing values in `last_purchase_days_ago`.
2.  **Data Splitting:** Split the dataset into training and testing sets.
3.  **Preprocessing Pipeline:** Construct a robust preprocessing pipeline using `ColumnTransformer` to handle:
    *   Missing values in `last_purchase_days_ago` (e.g., using median imputation).
    *   Categorical features (`membership_tier`, `region`) using one-hot encoding.
    *   Numerical features (`age`, `total_purchases`, `avg_transaction_value`, `last_purchase_days_ago`) using standard scaling.
4.  **Model Selection:** Choose an appropriate regression model (e.g., `RandomForestRegressor`, `GradientBoostingRegressor`, `XGBoostRegressor` if installed).
5.  **Full Pipeline Construction:** Combine your preprocessing steps and the chosen model into a single `scikit-learn` Pipeline.
6.  **Hyperparameter Tuning:** Use `GridSearchCV` or `RandomizedSearchCV` to find the best hyperparameters for your model within the pipeline. Focus on a few key parameters to keep computation reasonable.
7.  **Model Training and Evaluation:** Train the best model from your hyperparameter search on the training data and evaluate its performance on the test set using appropriate regression metrics (e.g., Mean Absolute Error (MAE), Root Mean Squared Error (RMSE), R-squared).
8.  **Interpretation:** Briefly discuss the performance of your model and any insights you gained. What are potential next steps or improvements?

**Deliverables:**
*   Well-commented Python code demonstrating all the steps above.
*   Printed evaluation metrics for the final model.
*   A brief markdown cell discussing your findings.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# --- 1. Data Generation (Starter Code) ---
np.random.seed(42)
num_samples = 1000

data = {
    'age': np.random.randint(18, 70, num_samples),
    'total_purchases': np.random.randint(1, 100, num_samples),
    'avg_transaction_value': np.random.uniform(10, 500, num_samples),
    'membership_tier': np.random.choice(['Bronze', 'Silver', 'Gold', 'Platinum'], num_samples, p=[0.4, 0.3, 0.2, 0.1]),
    'last_purchase_days_ago': np.random.normal(30, 20, num_samples).clip(0, 180),
    'region': np.random.choice(['North', 'South', 'East', 'West'], num_samples, p=[0.25, 0.25, 0.25, 0.25])
}

df = pd.DataFrame(data)

# Introduce some missing values in 'last_purchase_days_ago'
missing_indices = np.random.choice(df.index, int(num_samples * 0.1), replace=False)
df.loc[missing_indices, 'last_purchase_days_ago'] = np.nan

# Generate CLV (target variable) with some noise and feature dependency
df['clv'] = (df['total_purchases'] * df['avg_transaction_value'] * 0.5 +
             df['age'] * 5 + 
             df['membership_tier'].map({'Bronze': 100, 'Silver': 200, 'Gold': 400, 'Platinum': 800}) +
             (180 - df['last_purchase_days_ago'].fillna(90)) * 2 + 
             np.random.normal(0, 200, num_samples))

# Ensure CLV is non-negative
df['clv'] = df['clv'].clip(lower=0)

print("Dataset Head:")
print(df.head())
print("\nMissing values:")
print(df.isnull().sum())

# --- 2. Data Splitting ---
X = df.drop('clv', axis=1)
y = df['clv']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"\nTraining set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

# --- 3. Preprocessing Pipeline (Your Task: Define transformers) ---

# Define numerical and categorical features
numerical_features = ['age', 'total_purchases', 'avg_transaction_value', 'last_purchase_days_ago']
categorical_features = ['membership_tier', 'region']

# Create preprocessing steps for numerical features
numerical_transformer = Pipeline(steps=[
    # TODO: Add an imputer for missing values
    # TODO: Add a scaler for numerical features
])

# Create preprocessing steps for categorical features
categorical_transformer = Pipeline(steps=[
    # TODO: Add an encoder for categorical features
])

# Create a ColumnTransformer to apply different transformations to different columns
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# --- 4. Model Selection (Your Task: Choose a model) ---
# model = # TODO: Instantiate your chosen regression model here

# --- 5. Full Pipeline Construction (Your Task: Combine preprocessor and model) ---
# full_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
#                                 ('regressor', model)])

# --- 6. Hyperparameter Tuning (Your Task: Define param_grid and GridSearchCV) ---
# param_grid = {
#     # TODO: Define hyperparameters to tune for your chosen model
# }

# grid_search = GridSearchCV(full_pipeline, param_grid, cv=5, scoring='neg_mean_squared_error', n_jobs=-1, verbose=1)

# --- 7. Model Training and Evaluation (Your Task: Fit and evaluate) ---
# grid_search.fit(X_train, y_train)

# best_model = grid_search.best_estimator_

# y_pred = best_model.predict(X_test)

# mae = mean_absolute_error(y_test, y_pred)
# rmse = np.sqrt(mean_squared_error(y_test, y_pred))
# r2 = r2_score(y_test, y_pred)

# print(f"\nBest parameters: {grid_search.best_params_}")
# print(f"Mean Absolute Error (MAE): {mae:.2f}")
# print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
# print(f"R-squared (R2): {r2:.2f}")

# --- 8. Interpretation (Your Task: Add a markdown cell below with your discussion) ---


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# --- 1. Data Generation ---
np.random.seed(42)
num_samples = 1000

data = {
    'age': np.random.randint(18, 70, num_samples),
    'total_purchases': np.random.randint(1, 100, num_samples),
    'avg_transaction_value': np.random.uniform(10, 500, num_samples),
    'membership_tier': np.random.choice(['Bronze', 'Silver', 'Gold', 'Platinum'], num_samples, p=[0.4, 0.3, 0.2, 0.1]),
    'last_purchase_days_ago': np.random.normal(30, 20, num_samples).clip(0, 180),
    'region': np.random.choice(['North', 'South', 'East', 'West'], num_samples, p=[0.25, 0.25, 0.25, 0.25])
}

df = pd.DataFrame(data)

# Introduce some missing values in 'last_purchase_days_ago'
missing_indices = np.random.choice(df.index, int(num_samples * 0.1), replace=False)
df.loc[missing_indices, 'last_purchase_days_ago'] = np.nan

# Generate CLV (target variable) with some noise and feature dependency
# CLV is influenced by purchase volume, value, age, membership tier, and recency of last purchase
df['clv'] = (df['total_purchases'] * df['avg_transaction_value'] * 0.5 +
             df['age'] * 5 + 
             df['membership_tier'].map({'Bronze': 100, 'Silver': 200, 'Gold': 400, 'Platinum': 800}) +
             (180 - df['last_purchase_days_ago'].fillna(90)) * 2 + 
             np.random.normal(0, 200, num_samples))

# Ensure CLV is non-negative
df['clv'] = df['clv'].clip(lower=0)

print("Dataset Head:")
print(df.head())
print("\nMissing values:")
print(df.isnull().sum())

# --- 2. Data Splitting ---
X = df.drop('clv', axis=1)
y = df['clv']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"\nTraining set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

# --- 3. Preprocessing Pipeline ---

# Define numerical and categorical features
numerical_features = ['age', 'total_purchases', 'avg_transaction_value', 'last_purchase_days_ago']
categorical_features = ['membership_tier', 'region']

# Create preprocessing steps for numerical features
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')), # Impute missing values with the median
    ('scaler', StandardScaler()) # Scale numerical features
])

# Create preprocessing steps for categorical features
categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore')) # One-hot encode categorical features
])

# Create a ColumnTransformer to apply different transformations to different columns
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# --- 4. Model Selection ---
# We'll use RandomForestRegressor as a robust choice for this problem.
model = RandomForestRegressor(random_state=42)

# --- 5. Full Pipeline Construction ---
full_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                ('regressor', model)])

# --- 6. Hyperparameter Tuning ---
# Define a parameter grid for GridSearchCV. We'll tune a few key parameters for RandomForestRegressor.
param_grid = {
    'regressor__n_estimators': [100, 200], # Number of trees in the forest
    'regressor__max_depth': [10, 20, None], # Maximum depth of the tree
    'regressor__min_samples_split': [2, 5] # Minimum number of samples required to split an internal node
}

# Initialize GridSearchCV
# Using 'neg_mean_squared_error' as scoring, as GridSearchCV maximizes scores.
# n_jobs=-1 uses all available CPU cores.
grid_search = GridSearchCV(full_pipeline, param_grid, cv=3, scoring='neg_mean_squared_error', n_jobs=-1, verbose=1)

# --- 7. Model Training and Evaluation ---
print("\nStarting GridSearchCV...")
grid_search.fit(X_train, y_train)

# Get the best model from the grid search
best_model = grid_search.best_estimator_

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate evaluation metrics
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"\nBest parameters found: {grid_search.best_params_}")
print(f"Mean Absolute Error (MAE): {mae:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
print(f"R-squared (R2): {r2:.2f}")

# --- 8. Interpretation (as a markdown cell below) ---


## Part 2: Capstone Project - Interpretation

**Model Performance Discussion:**

The Random Forest Regressor model, after hyperparameter tuning with `GridSearchCV`, achieved the following performance on the test set:
*   **Mean Absolute Error (MAE):** [Value from output] - This indicates, on average, how much our predictions deviate from the actual CLV values. A lower MAE is better.
*   **Root Mean Squared Error (RMSE):** [Value from output] - Similar to MAE, but penalizes larger errors more heavily. A lower RMSE is better.
*   **R-squared (R2):** [Value from output] - This metric represents the proportion of the variance in the dependent variable that is predictable from the independent variables. An R2 close to 1 indicates a good fit, while 0 indicates the model explains no variance.

Given the synthetic nature of the data, the model performed reasonably well, demonstrating its ability to capture the underlying relationships between customer features and CLV. The `RandomForestRegressor` is a robust choice for this type of problem, handling both numerical and categorical features effectively through the preprocessing pipeline.

**Insights and Next Steps:**

1.  **Feature Importance:** While not explicitly calculated in this assessment, analyzing feature importance from the best `RandomForestRegressor` would provide insights into which customer attributes are most predictive of CLV. This could inform marketing strategies.
2.  **More Data & Features:** In a real-world scenario, incorporating more diverse data (e.g., website activity, customer service interactions, product categories purchased) could significantly improve model accuracy.
3.  **Advanced Models:** Exploring more advanced models like XGBoost or LightGBM could potentially yield better performance, especially with larger datasets.
4.  **Time-Series Aspects:** CLV inherently has a time-series component. For a more sophisticated model, incorporating time-series features (e.g., recency, frequency, monetary value - RFM analysis) and potentially using time-series specific modeling techniques could be beneficial.
5.  **Deployment Considerations:** Once satisfied with the model's performance, the next step would be to serialize the `best_model` (which includes the entire preprocessing pipeline) using `joblib` or `pickle` for deployment into a production environment, allowing real-time CLV predictions for new customers.
